# Historical Energy Demand Analysis
Read UNSD country-level energy balance CSVs, convert TJ → TWh, and explore demand by sector and subsector.

In [2]:
from pathlib import Path
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Resolve workspace root robustly (works regardless of kernel cwd) ───────────
_THIS_FILE = Path(os.path.abspath("")) # kernel cwd
# Walk up until we find the known marker folder "data/country_specific"
def _find_workspace_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "data" / "country_specific").exists():
            return parent
    raise RuntimeError(
        f"Cannot locate 'data/country_specific' starting from {start}. "
        "Make sure the workspace root is correct."
    )

WORKSPACE_ROOT = _find_workspace_root(_THIS_FILE)
DATA_DIR = WORKSPACE_ROOT / "data" / "country_specific"
print(f"Workspace root : {WORKSPACE_ROOT}")
print(f"Data directory : {DATA_DIR}  (exists={DATA_DIR.exists()})")

# ── Configuration ──────────────────────────────────────────────────────────────
COUNTRY = "Namibia"          # change to any country available in data/country_specific
TJ_TO_TWH = 0.000277778

# ── Sector / subsector mapping ─────────────────────────────────────────────────
SECTOR_MAP = {
    "Industry": {
        "row": "Manufacturing  const. and mining",
        "subsectors": [
            "Iron and steel",
            "Chemical and petrochemical",
            "Non-ferrous metals",
            "Non-metallic minerals",
            "Transport equipment",
            "Machinery",
            "Mining and quarrying",
            "Food and tobacco",
            "Paper  pulp and printing",
            "Wood and wood products",
            "Textile and leather",
            "Construction",
            "Industry n.e.s",
        ],
    },
    "Transport": {
        "row": "Transport",
        "subsectors": [
            "Road",
            "Rail",
            "Domestic aviation",
            "Domestic navigation",
            "Pipeline transport",
            "Transport n.e.s",
        ],
    },
    "Residential": {
        "row": "Households",
        "subsectors": [],
    },
    "Agriculture": {
        "row": "Agriculture  forestry and fishing",
        "subsectors": [],
    },
    "Services": {
        "row": "Commerce and public services",
        "subsectors": [],
    },
    "Non-energy use": {
        "row": "Non-energy use",
        "subsectors": [],
    },
    "Other": {
        "row": "Other consumption n.e.s",
        "subsectors": [],
    },
}

FUEL_COLS = [
    "Primary Coal and Peat",
    "Coal and Peat Products",
    "Primary Oil",
    "Oil Products",
    "Natural Gas",
    "Biofuels and Waste",
    "Nuclear",
    "Electricity",
    "Heat",
]

print(f"Config loaded. Country: {COUNTRY}")


Workspace root : c:\Users\eduque\Documents\models\sectoral-energy-demands
Data directory : c:\Users\eduque\Documents\models\sectoral-energy-demands\data\country_specific  (exists=True)
Config loaded. Country: Namibia


## 1 · Load & clean all yearly files

In [3]:
def _clean_val(raw) -> float:
    """Convert a UNSD cell value (may be '---', '*123', etc.) to float TJ."""
    if pd.isna(raw):
        return 0.0
    s = str(raw).strip().lstrip("*").replace(",", "")
    try:
        return float(s)
    except ValueError:
        return 0.0


def load_country_years(country: str, data_dir: Path) -> dict[int, pd.DataFrame]:
    """Return {year: raw_df} for all years found for the given country."""
    files = sorted(data_dir.glob(f"UNSD_{country}-*.csv"))
    if not files:
        raise FileNotFoundError(f"No files found for '{country}' in {data_dir}")
    result = {}
    for f in files:
        year = int(f.stem.split("-")[-1])
        df = pd.read_csv(f)
        # Normalise the transaction column name (first non-country col)
        df.columns = [c.strip() for c in df.columns]
        result[year] = df
    print(f"Loaded {len(result)} years for {country}: {sorted(result)}")
    return result


def get_row(df: pd.DataFrame, label: str) -> pd.Series | None:
    """Return the first row whose transaction label matches (case/space insensitive)."""
    col = "Transactions(down)/Commodity(right)"
    if col not in df.columns:
        return None
    norm = lambda s: " ".join(str(s).strip().lower().split())
    mask = df[col].apply(norm) == norm(label)
    rows = df[mask]
    return rows.iloc[0] if not rows.empty else None


def row_to_twh(row: pd.Series, fuel_cols: list[str]) -> dict:
    """Extract TJ values for each fuel, return as TWh dict."""
    return {
        fuel: round(_clean_val(row.get(fuel, 0)) * TJ_TO_TWH, 6)
        for fuel in fuel_cols
    }


raw_years = load_country_years(COUNTRY, DATA_DIR)

Loaded 13 years for Namibia: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


## 2 · Build tidy DataFrames — sector totals & subsector breakdown

In [4]:
sector_rows = []    # one row per (year, sector)
subsector_rows = [] # one row per (year, sector, subsector)

for year, df in sorted(raw_years.items()):
    for sector_name, cfg in SECTOR_MAP.items():
        row = get_row(df, cfg["row"])
        if row is None:
            continue

        fuels_twh = row_to_twh(row, FUEL_COLS)
        total_twh = round(sum(fuels_twh.values()), 6)
        if total_twh == 0:
            continue

        sector_rows.append({
            "year": year,
            "sector": sector_name,
            "total_TWh": total_twh,
            **fuels_twh,
        })

        for sub_label in cfg["subsectors"]:
            sub_row = get_row(df, sub_label)
            if sub_row is None:
                continue
            sub_fuels = row_to_twh(sub_row, FUEL_COLS)
            sub_total = round(sum(sub_fuels.values()), 6)
            if sub_total == 0:
                continue
            subsector_rows.append({
                "year": year,
                "sector": sector_name,
                "subsector": sub_label,
                "total_TWh": sub_total,
                **sub_fuels,
            })

df_sectors = pd.DataFrame(sector_rows).sort_values(["year", "sector"]).reset_index(drop=True)
df_subsectors = pd.DataFrame(subsector_rows).sort_values(["year", "sector", "subsector"]).reset_index(drop=True)

print(f"Sector records:    {len(df_sectors)}")
print(f"Subsector records: {len(df_subsectors)}")
df_sectors.tail(15)

Sector records:    91
Subsector records: 98


,year,sector,total_TWh,Primary Coal and Peat,Coal and Peat Products,Primary Oil,Oil Products,Natural Gas,Biofuels and Waste,Nuclear,Electricity,Heat
76,2021,Transport,7.280284,0.0,0.0,0.0,7.280284,0.0,0.000000,0.0,0.000000,0.000000
77,2022,Agriculture,0.759167,0.0,0.0,0.0,0.560834,0.0,0.000000,0.0,0.198333,0.000000
78,2022,Industry,2.585279,0.0,0.0,0.0,0.802778,0.0,0.426389,0.0,1.356112,0.000000
79,2022,Non-energy use,0.205278,0.0,0.0,0.0,0.205278,0.0,0.000000,0.0,0.000000,0.000000
80,2022,Other,0.448056,0.0,0.0,0.0,0.090278,0.0,0.000000,0.0,0.301111,0.056667
81,2022,Residential,3.007780,0.0,0.0,0.0,0.024167,0.0,1.967779,0.0,1.015834,0.000000
82,2022,Services,3.840558,0.0,0.0,0.0,2.906391,0.0,0.570278,0.0,0.363889,0.000000
83,2022,Transport,7.037783,0.0,0.0,0.0,7.037783,0.0,0.000000,0.0,0.000000,0.000000
84,2023,Agriculture,0.792778,0.0,0.0,0.0,0.586389,0.0,0.000000,0.0,0.206389,0.000000
85,2023,Industry,2.840836,0.0,0.0,0.0,0.875834,0.0,0.453334,0.0,1.511668,0.000000


## 3 · Summary table — latest year

In [5]:
latest_year = df_sectors["year"].max()

# Sector summary
sector_summary = (
    df_sectors[df_sectors["year"] == latest_year]
    [["sector", "total_TWh"] + FUEL_COLS]
    .set_index("sector")
    .sort_values("total_TWh", ascending=False)
)
sector_summary.columns.name = f"{COUNTRY} — {latest_year} (TWh)"
print(f"\n=== Sector totals {latest_year} ===")
display(sector_summary.style.format("{:.3f}").background_gradient(cmap="YlOrRd", axis=None))

# Subsector detail for the same year
sub_latest = (
    df_subsectors[df_subsectors["year"] == latest_year]
    [["sector", "subsector", "total_TWh"] + FUEL_COLS]
    .sort_values(["sector", "total_TWh"], ascending=[True, False])
    .set_index(["sector", "subsector"])
)
print(f"\n=== Subsector detail {latest_year} ===")
display(sub_latest.style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))


=== Sector totals 2023 ===


Namibia — 2023 (TWh),total_TWh,Primary Coal and Peat,Coal and Peat Products,Primary Oil,Oil Products,Natural Gas,Biofuels and Waste,Nuclear,Electricity,Heat
sector,,,,,,,,,,
Transport,7.333,0.000,0.000,0.000,7.333,0.000,0.000,0.000,0.000,0.000
Services,4.001,0.000,0.000,0.000,3.027,0.000,0.594,0.000,0.379,0.000
Residential,3.054,0.000,0.000,0.000,0.025,0.000,1.999,0.000,1.030,0.000
Industry,2.841,0.000,0.000,0.000,0.876,0.000,0.453,0.000,1.512,0.000
Agriculture,0.793,0.000,0.000,0.000,0.586,0.000,0.000,0.000,0.206,0.000
Other,0.469,0.000,0.000,0.000,0.099,0.000,0.000,0.000,0.314,0.057
Non-energy use,0.210,0.000,0.000,0.000,0.210,0.000,0.000,0.000,0.000,0.000



=== Subsector detail 2023 ===


## 4 · Historical trend — total demand by sector (stacked area)

In [6]:
fig = px.area(
    df_sectors,
    x="year",
    y="total_TWh",
    color="sector",
    title=f"{COUNTRY} — Historical Final Energy Demand by Sector (TWh)",
    labels={"total_TWh": "TWh", "year": "Year", "sector": "Sector"},
    template="plotly_white",
)
fig.update_layout(hovermode="x unified", legend_title_text="Sector")
fig.show()

## 5 · Industry subsector breakdown over time

In [7]:
df_industry_sub = df_subsectors[df_subsectors["sector"] == "Industry"]

if df_industry_sub.empty:
    print("No industry subsector data available for this country.")
else:
    fig = px.bar(
        df_industry_sub,
        x="year",
        y="total_TWh",
        color="subsector",
        barmode="stack",
        title=f"{COUNTRY} — Industry Subsector Breakdown (TWh)",
        labels={"total_TWh": "TWh", "year": "Year", "subsector": "Subsector"},
        template="plotly_white",
    )
    fig.update_layout(hovermode="x unified", legend_title_text="Subsector")
    fig.show()

## 6 · Transport subsector breakdown over time

In [8]:
df_transport_sub = df_subsectors[df_subsectors["sector"] == "Transport"]

if df_transport_sub.empty:
    print("No transport subsector data available for this country.")
else:
    fig = px.bar(
        df_transport_sub,
        x="year",
        y="total_TWh",
        color="subsector",
        barmode="stack",
        title=f"{COUNTRY} — Transport Subsector Breakdown (TWh)",
        labels={"total_TWh": "TWh", "year": "Year", "subsector": "Subsector"},
        template="plotly_white",
    )
    fig.update_layout(hovermode="x unified", legend_title_text="Subsector")
    fig.show()

## 7 · Fuel mix per sector (latest year — grouped bar)

In [12]:
df_latest = df_sectors[df_sectors["year"] == 2020].copy()

# Melt fuel columns into long format
df_fuels_long = df_latest.melt(
    id_vars=["sector"],
    value_vars=FUEL_COLS,
    var_name="fuel",
    value_name="TWh",
)
df_fuels_long = df_fuels_long[df_fuels_long["TWh"] > 0]

fig = px.bar(
    df_fuels_long,
    x="sector",
    y="TWh",
    color="fuel",
    barmode="stack",
    title=f"{COUNTRY} — Fuel Mix by Sector in {latest_year} (TWh)",
    labels={"TWh": "TWh", "sector": "Sector", "fuel": "Fuel"},
    template="plotly_white",
)
fig.update_layout(legend_title_text="Fuel carrier")
fig.show()

## 8 · Per-sector line charts with fuel breakdown

In [10]:
sectors_with_data = df_sectors["sector"].unique().tolist()
n = len(sectors_with_data)
cols = 2
rows_n = (n + 1) // cols

fig = make_subplots(
    rows=rows_n, cols=cols,
    subplot_titles=sectors_with_data,
    shared_xaxes=False,
)

for idx, sector in enumerate(sectors_with_data):
    r = idx // cols + 1
    c = idx % cols + 1
    df_s = df_sectors[df_sectors["sector"] == sector]
    df_melt = df_s.melt(id_vars=["year"], value_vars=FUEL_COLS, var_name="fuel", value_name="TWh")
    df_melt = df_melt[df_melt["TWh"] > 0]

    for fuel, grp in df_melt.groupby("fuel"):
        fig.add_trace(
            go.Scatter(
                x=grp["year"], y=grp["TWh"],
                mode="lines+markers",
                name=fuel,
                legendgroup=fuel,
                showlegend=(idx == 0),
                line=dict(width=2),
            ),
            row=r, col=c,
        )

fig.update_layout(
    height=300 * rows_n,
    title_text=f"{COUNTRY} — Energy Demand by Fuel per Sector (TWh)",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()